# Add tourism + finalize AI readiness 2025 (data update notebook)

**Goal.** Extend the harmonised GCC panel with **two tourism series** and complete the **2025 AI readiness** headline score.

**Steps.**
1. Reshape `datasets/tourism.csv` (WDI wide format) to a long panel with two columns:
   - `tourism_receipts_pct_exports` — `ST.INT.RCPT.XP.ZS` (international tourism receipts, % of total exports)
   - `tourism_receipts_usd` — `ST.INT.RCPT.CD` (international tourism receipts, current US$)
2. Merge into `datasets/merged_dataset.csv` on `(country, year)` and insert the new columns after `export_concentration` (external-sector grouping).
3. Fill `ai_readiness` for **2025** rows with the **mean of the six 2025 pillars** (so the headline score column is complete across editions).
4. Persist:
   - Overwrite `datasets/merged_dataset.csv` (replaces the previous file).
   - Update `datasets/postoil.db`: add `tourism` table; refresh `ai_readiness` (2025 rows now carry the computed score).

**Coverage notes (already inspected).**
- `tourism_receipts_pct_exports` is fully missing for **United Arab Emirates** (will remain NaN there); other GCC have multi-year coverage.
- `tourism_receipts_usd` is available for all six GCC including UAE.
- 2025: tourism columns are blank for all GCC in this WDI extract.

In [1]:
import os
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../datasets")
TOURISM_CSV = DATA_DIR / "tourism.csv"
MERGED_CSV = DATA_DIR / "merged_dataset.csv"
DB_PATH = DATA_DIR / "postoil.db"

GCC_COUNTRIES = [
    "Bahrain", "Kuwait", "Oman", "Qatar", "Saudi Arabia", "United Arab Emirates",
]

CODE_TO_NAME = {
    "BHR": "Bahrain",
    "KWT": "Kuwait",
    "OMN": "Oman",
    "QAT": "Qatar",
    "SAU": "Saudi Arabia",
    "ARE": "United Arab Emirates",
}

print("Files exist:")
for p in [TOURISM_CSV, MERGED_CSV, DB_PATH]:
    print(f"  {p}: {p.exists()}")

Files exist:
  ../datasets/tourism.csv: True
  ../datasets/merged_dataset.csv: True
  ../datasets/postoil.db: True


### 1) Reshape `tourism.csv` to long format

The WDI export is in wide format with year columns like `"2023 [YR2023]"`. We:
- keep only GCC rows,
- map series codes to clean column names,
- pivot to `(country, year)` with two value columns,
- coerce `..` (World Bank "missing") to `NaN`.

In [2]:
SERIES_TO_COLUMN = {
    "ST.INT.RCPT.XP.ZS": "tourism_receipts_pct_exports",
    "ST.INT.RCPT.CD": "tourism_receipts_usd",
}

raw = pd.read_csv(TOURISM_CSV)
raw = raw[raw["Country Code"].isin(CODE_TO_NAME.keys())].copy()
raw["country"] = raw["Country Code"].map(CODE_TO_NAME)

year_cols = [c for c in raw.columns if re.match(r"^\d{4}\s*\[YR\d{4}\]$", str(c))]

long_df = (
    raw.melt(
        id_vars=["country", "Series Code"],
        value_vars=year_cols,
        var_name="year_label",
        value_name="value",
    )
)
long_df["year"] = long_df["year_label"].str.extract(r"^(\d{4})").astype(int)
long_df["value"] = pd.to_numeric(long_df["value"].replace("..", np.nan), errors="coerce")
long_df["column"] = long_df["Series Code"].map(SERIES_TO_COLUMN)

long_df = long_df.dropna(subset=["column"]).copy()

tourism = (
    long_df.pivot_table(
        index=["country", "year"],
        columns="column",
        values="value",
        aggfunc="first",
    )
    .reset_index()
)

tourism.columns.name = None
for col in SERIES_TO_COLUMN.values():
    if col not in tourism.columns:
        tourism[col] = np.nan

tourism = tourism[["country", "year", "tourism_receipts_pct_exports", "tourism_receipts_usd"]]
print("Tourism long shape:", tourism.shape)
print("\nNon-null counts by country (full span):")
print(tourism.groupby("country")[["tourism_receipts_pct_exports", "tourism_receipts_usd"]].apply(lambda d: d.notna().sum()))
print("\nLast 8 years preview (Saudi Arabia):")
print(tourism[tourism["country"] == "Saudi Arabia"].tail(8))

Tourism long shape: (126, 4)

Non-null counts by country (full span):
                      tourism_receipts_pct_exports  tourism_receipts_usd
country                                                                 
Bahrain                                         26                    26
Kuwait                                          26                    26
Oman                                            23                    23
Qatar                                           10                    10
Saudi Arabia                                    15                    15
United Arab Emirates                             0                    26

Last 8 years preview (Saudi Arabia):
         country  year  tourism_receipts_pct_exports  tourism_receipts_usd
92  Saudi Arabia  2013                      2.235936          8.690000e+09
93  Saudi Arabia  2014                      2.602865          9.263000e+09
94  Saudi Arabia  2015                      5.108060          1.118300e+10
95  Saud

### 2) Merge tourism into `merged_dataset.csv`

We left-join on `(country, year)` and place tourism columns **right after `export_concentration`** to keep external-sector indicators grouped (concentration → tourism share → tourism level → exports/imports USD → GDP).

In [3]:
data = pd.read_csv(MERGED_CSV)

before_cols = list(data.columns)
print("Original column count:", len(before_cols))

# Drop existing tourism columns if a previous run already inserted them (idempotent).
data = data.drop(columns=[c for c in ["tourism_receipts_pct_exports", "tourism_receipts_usd"] if c in data.columns])

merged = data.merge(tourism, on=["country", "year"], how="left")

# Re-order: insert tourism columns right after export_concentration.
ordered = []
for c in merged.columns:
    if c in {"tourism_receipts_pct_exports", "tourism_receipts_usd"}:
        continue
    ordered.append(c)
    if c == "export_concentration":
        ordered.append("tourism_receipts_pct_exports")
        ordered.append("tourism_receipts_usd")

merged = merged[ordered]

print("New column count:", len(merged.columns))
print("New columns:", [c for c in merged.columns if "tourism" in c])
print("\nMerged shape:", merged.shape)
print("\nSample (Saudi Arabia, 2018-2023):")
print(
    merged[(merged["country"] == "Saudi Arabia") & (merged["year"].between(2018, 2023))][
        ["country", "year", "export_concentration", "tourism_receipts_pct_exports", "tourism_receipts_usd"]
    ]
)

Original column count: 27
New column count: 29
New columns: ['tourism_receipts_pct_exports', 'tourism_receipts_usd']

Merged shape: (364, 29)

Sample (Saudi Arabia, 2018-2023):
          country  year  export_concentration  tourism_receipts_pct_exports  \
300  Saudi Arabia  2018                 0.597                      5.372328   
301  Saudi Arabia  2019                 0.602                      6.850091   
302  Saudi Arabia  2020                 0.530                      3.223965   
303  Saudi Arabia  2021                 0.540                           NaN   
304  Saudi Arabia  2022                 0.596                           NaN   
305  Saudi Arabia  2023                 0.588                           NaN   

     tourism_receipts_usd  
300          1.697400e+10  
301          1.984900e+10  
302          5.960000e+09  
303                   NaN  
304                   NaN  
305                   NaN  


### 3) Complete `ai_readiness` for 2025 (mean of six 2025 pillars)

In `ai_readiness.csv`, 2025 rows have a blank `total_score` but the **six 2025 pillars** are populated:

`policy_capacity`, `ai_infrastructure`, `governance`, `public_sector_adoption`, `development_diffusion`, `resilience`.

The plotting rule already used in `main.ipynb` for 2025 is **mean of six pillars**. We persist that mean directly into `ai_readiness` for 2025 rows so downstream code (and the database) sees a complete column across editions.

In [4]:
PILLARS_2025 = [
    "policy_capacity",
    "ai_infrastructure",
    "governance",
    "public_sector_adoption",
    "development_diffusion",
    "resilience",
]

mask_2025 = merged["year"] == 2025
mean_2025 = merged.loc[mask_2025, PILLARS_2025].mean(axis=1)

# Only fill where ai_readiness is missing for 2025 rows (preserve any existing value).
to_fill = mask_2025 & merged["ai_readiness"].isna() & mean_2025.notna()
merged.loc[to_fill, "ai_readiness"] = mean_2025[to_fill].round(4)

print("2025 rows updated:", int(to_fill.sum()))
print("\nResulting 2025 AI readiness:")
print(
    merged[mask_2025][["country", "year", "ai_readiness"] + PILLARS_2025]
    .sort_values("ai_readiness", ascending=False)
    .reset_index(drop=True)
)

2025 rows updated: 6

Resulting 2025 AI readiness:
                country  year  ai_readiness  policy_capacity  \
0          Saudi Arabia  2025       74.7717             92.0   
1  United Arab Emirates  2025       69.5050             73.0   
2                 Qatar  2025       60.3767             80.5   
3                  Oman  2025       60.0167             88.5   
4               Bahrain  2025       59.6650             46.5   
5                Kuwait  2025       43.4283             35.0   

   ai_infrastructure  governance  public_sector_adoption  \
0              62.93       93.21                   88.62   
1              73.77       72.50                   97.27   
2              57.95       77.50                   62.82   
3              47.22       72.00                   71.00   
4              58.04       72.50                   83.37   
5              53.31       52.63                   51.43   

   development_diffusion  resilience  
0                  54.48       57.39  
1

### 4) Validation snapshot before persisting

Quick sanity prints: shape, panel integrity, NaN counts on the new tourism columns by country, and a coverage table for the focus window 2010-2024.

In [5]:
assert merged.duplicated(subset=["country", "year"]).sum() == 0, "Duplicate (country, year) rows"
assert set(merged["country"].unique()) == set(GCC_COUNTRIES), "Country set mismatch"

print("Final merged shape:", merged.shape)
print("\nGlobal NaN counts (tourism + ai_readiness):")
print(merged[["tourism_receipts_pct_exports", "tourism_receipts_usd", "ai_readiness"]].isna().sum())

print("\nNon-null tourism rows by country (full panel):")
print(
    merged.groupby("country")[["tourism_receipts_pct_exports", "tourism_receipts_usd"]].apply(
        lambda d: d.notna().sum()
    )
)

focus = merged[merged["year"].between(2010, 2024)].copy()
print("\nCoverage 2010-2024 — share with both tourism columns present:")
print(
    focus.groupby("country")[["tourism_receipts_pct_exports", "tourism_receipts_usd"]]
    .apply(lambda d: d.notna().all(axis=1).sum())
    .rename("rows_complete_both")
)

Final merged shape: (364, 29)

Global NaN counts (tourism + ai_readiness):
tourism_receipts_pct_exports    264
tourism_receipts_usd            238
ai_readiness                    346
dtype: int64

Non-null tourism rows by country (full panel):
                      tourism_receipts_pct_exports  tourism_receipts_usd
country                                                                 
Bahrain                                         26                    26
Kuwait                                          26                    26
Oman                                            23                    23
Qatar                                           10                    10
Saudi Arabia                                    15                    15
United Arab Emirates                             0                    26

Coverage 2010-2024 — share with both tourism columns present:
country
Bahrain                 11
Kuwait                  11
Oman                    11
Qatar               

### 5) Persist — overwrite CSV + update SQLite DB

- `datasets/merged_dataset.csv` is **overwritten** (replaces the previous file as requested).
- `datasets/postoil.db`:
  - new table `tourism (country TEXT, year INTEGER, tourism_receipts_pct_exports REAL, tourism_receipts_usd REAL)`
  - existing `ai_readiness` table is refreshed so 2025 rows now carry the computed `ai_readiness` value (column name `ai_readiness` in the table).

In [6]:
merged.to_csv(MERGED_CSV, index=False)
print(f"Wrote {MERGED_CSV} ({merged.shape[0]} rows, {merged.shape[1]} columns)")

with sqlite3.connect(DB_PATH) as con:
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS tourism")
    cur.execute(
        """
        CREATE TABLE tourism (
            country TEXT,
            year INTEGER,
            tourism_receipts_pct_exports REAL,
            tourism_receipts_usd REAL
        )
        """
    )
    tourism_db = (
        merged[["country", "year", "tourism_receipts_pct_exports", "tourism_receipts_usd"]]
        .dropna(subset=["tourism_receipts_pct_exports", "tourism_receipts_usd"], how="all")
        .copy()
    )
    cur.executemany(
        "INSERT INTO tourism VALUES (?, ?, ?, ?)",
        tourism_db.itertuples(index=False, name=None),
    )

    # Refresh ai_readiness rows for 2025 with the computed mean.
    ai_2025 = merged[merged["year"] == 2025][
        [
            "country",
            "year",
            "ai_readiness",
            "ai_readiness_rank",
            "government_pillar",
            "technology_sector_pillar",
            "data_infrastructure_pillar",
            "policy_capacity",
            "ai_infrastructure",
            "governance",
            "public_sector_adoption",
            "development_diffusion",
            "resilience",
        ]
    ]
    cur.execute("DELETE FROM ai_readiness WHERE year = 2025")
    cur.executemany(
        """
        INSERT INTO ai_readiness (
            country, year, ai_readiness, ai_readiness_rank,
            government_pillar, technology_sector_pillar, data_infrastructure_pillar,
            policy_capacity, ai_infrastructure, governance,
            public_sector_adoption, development_diffusion, resilience
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        ai_2025.itertuples(index=False, name=None),
    )

    con.commit()

    print("\nTables in postoil.db:")
    for (name,) in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
        n = cur.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"  {name}: {n} rows")

    print("\nai_readiness 2025 rows from DB:")
    for row in cur.execute(
        "SELECT country, year, ai_readiness FROM ai_readiness WHERE year = 2025 ORDER BY ai_readiness DESC"
    ):
        print("  ", row)

Wrote ../datasets/merged_dataset.csv (364 rows, 29 columns)

Tables in postoil.db:


  ai_readiness: 18 rows

ai_readiness 2025 rows from DB:
   ('Saudi Arabia', 2025, 74.7717)
   ('United Arab Emirates', 2025, 69.505)
   ('Qatar', 2025, 60.3767)
   ('Oman', 2025, 60.0167)
   ('Bahrain', 2025, 59.665)
   ('Kuwait', 2025, 43.4283)
